# Measuring a Classifier with More Than Two Classes

**CS 212 AI Programming 1**

This is the second of two Jupyter notebooks for the [Lab 4 example](https://lcc-cit.github.io/CS212-CourseMaterials/Examples/Lab04-ClassifierMetrics/CS212_Lab04_Example.html). The first one, `SpamMetrics.ipynb`, measures the spam classifier and explains accuracy, precision, recall, F1 and the confusion matrix. Read it first.

The spam classifier chooses between two classes. Of the three lab assignment versions, Group B's classifier chooses between six classes (six departments), so if you are in Group B, you will read a report like the one below. When there are more than two classes, scikit-learn calculates precision, recall and F1 for each class in turn, treating that class as the positive one.


## 1. The Data

To show what that looks like, this notebook uses the newsgroups example from the scikit-learn text tutorial in Week 3: posts from four newsgroups (computer graphics, hockey, medicine and religion), classified with the tf-idf and linear SVM pipeline from the tutorial. Two of the tutorial's newsgroups (atheism and Christianity) are replaced with hockey and religion. To show what happens when one class is rare, the code below throws away most of the religion posts: it keeps only 75 of the 377 for training and 50 of the 251 for testing. The other three newsgroups keep all of their posts, about 600 each for training and 390 each for testing. The first run downloads the data, so it needs an internet connection.

In [1]:
from sklearn.datasets import fetch_20newsgroups

categories = ["talk.religion.misc", "rec.sport.hockey", "comp.graphics", "sci.med"]
twenty_train = fetch_20newsgroups(subset="train", categories=categories, shuffle=True, random_state=42)
twenty_test = fetch_20newsgroups(subset="test", categories=categories, shuffle=True, random_state=42)
group_names = twenty_test.target_names
religion = group_names.index("talk.religion.misc")

# Keep only the first 75 of the 377 religion posts for training.
train_posts = []
train_groups = []
religion_count = 0
for i in range(len(twenty_train.data)):
    group = twenty_train.target[i]
    if group == religion:
        religion_count = religion_count + 1
    if group != religion or religion_count <= 75:
        train_posts.append(twenty_train.data[i])
        train_groups.append(group)

# Keep only the first 50 of the 251 religion posts for testing.
test_posts = []
test_groups = []
religion_count = 0
for i in range(len(twenty_test.data)):
    group = twenty_test.target[i]
    if group == religion:
        religion_count = religion_count + 1
    if group != religion or religion_count <= 50:
        test_posts.append(twenty_test.data[i])
        test_groups.append(group)

print(len(train_posts), "training posts and", len(test_posts), "test posts")

1853 training posts and 1234 test posts


## 2. The Classifier and Its Report

The classifier is the tf-idf and linear SVM pipeline from the tutorial. After it classifies the test posts, we print the classification report and the confusion matrix. In the confusion matrix, each row is a true newsgroup and each column is a predicted newsgroup, in the same order as the list of names printed above it.


In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

text_clf = Pipeline([
    ("vect", CountVectorizer()),
    ("tfidf", TfidfTransformer()),
    ("clf", SGDClassifier(loss="hinge", penalty="l2", alpha=1e-3,
                          random_state=42, max_iter=5, tol=None)),
])
text_clf.fit(train_posts, train_groups)
predicted_groups = text_clf.predict(test_posts)

print(classification_report(test_groups, predicted_groups, target_names=group_names))
print("Confusion matrix (rows = true, columns = predicted):")
print(group_names)
print(confusion_matrix(test_groups, predicted_groups))

                    precision    recall  f1-score   support

     comp.graphics       0.90      0.97      0.93       389
  rec.sport.hockey       0.97      0.99      0.98       399
           sci.med       0.93      0.90      0.92       396
talk.religion.misc       0.96      0.52      0.68        50

          accuracy                           0.94      1234
         macro avg       0.94      0.84      0.88      1234
      weighted avg       0.94      0.94      0.93      1234

Confusion matrix (rows = true, columns = predicted):
['comp.graphics', 'rec.sport.hockey', 'sci.med', 'talk.religion.misc']
[[377   3   9   0]
 [  3 394   2   0]
 [ 35   3 357   1]
 [  4   6  14  26]]


## 3. Reading the Report

Read the report one row at a time. For `talk.religion.misc`, the recall is 0.52: there were 50 religion posts in the test set and the classifier found only 26 of them. The precision is 0.96: the classifier put 27 posts in `talk.religion.misc` (0 + 0 + 1 + 26, the last column of the confusion matrix) and 26 of them belonged there. The other classes work the same way, each one taking its turn as the positive class. In the confusion matrix each row is a true newsgroup and each column is a predicted newsgroup, so the numbers on the diagonal are the correct answers.

At the bottom of the report are two kinds of average:

- **Macro average:** the plain average of the per-class scores. Every class counts the same, however small.
- **Weighted average:** the per-class scores averaged in proportion to the number of posts in each class (the *support* column).

Here the religion class is much smaller than the others: 50 test posts, compared with 389 to 399. In the weighted average it counts for only 50 of the 1234 posts, so the weighted average F1 is 0.93, close to the scores of the three big classes. In the macro average it counts as one class out of four, so its low recall pulls the macro average recall down to 0.84 and the macro average F1 down to 0.88. When the classes are not the same size, look at both averages and at each class's own row.

The accuracy of 0.94 hides the fact that the classifier misses about half of the religion posts. It is the same problem as the "always ham" classifier in the spam notebook: a rare class hardly changes the accuracy, so the accuracy can't tell you how well that class is classified. The confusion matrix shows where the missed posts went: 14 of the 24 were put in `sci.med`. With only 75 religion posts to learn from, the classifier rarely predicts that class. When it does, it is almost always right (high precision), but it misses many of the religion posts (low recall). The best fix is to collect more examples of the rare class. When you can't, you can at least choose a classifier's settings with a score that counts every class equally, as the next section shows.

## 4. Using a Metric to Tune the Classifier

Performance metrics are also how you decide whether a change made a classifier better. Most classifiers have *hyperparameters*, settings that you choose before training, such as the `alpha` of the classifier or whether the word counts include pairs of words. `GridSearchCV` tries every combination of the settings you give it. For each combination it trains and tests the classifier several times with cross-validation, using only the training posts. Then it keeps the combination with the best *score*.

That score is another performance metric, and you choose it with the `scoring` argument. The default for a classifier is accuracy, which has the problem described above: when one class is rare, accuracy can hardly tell a good classifier from a bad one. For a rare class you usually want `scoring="f1"`, which is the F1 of the positive class. For more than two classes, `scoring="f1_macro"` averages the F1 of every class, the same way the macro average in the report does.

The grid below tries three settings: whether to count word pairs, the `alpha` of the classifier, and `class_weight`. Normally every training post counts the same, so a class with only 75 posts has little effect on what the classifier learns. The setting `class_weight="balanced"` makes each post of a small class count for more, in proportion to how rare the class is. This time the two scores are accuracy and `f1_macro`, the average of the F1 of each newsgroup. To see how the two choices differ, both tuned classifiers are tested on the test posts. The cell takes about a minute to run.

In [3]:
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ("vect", CountVectorizer()),
    ("tfidf", TfidfTransformer()),
    ("clf", SGDClassifier(loss="hinge", penalty="l2", random_state=42, max_iter=5, tol=None)),
])
settings = {
    "vect__ngram_range": [(1, 1), (1, 2)],          # single words, or single words and word pairs
    "clf__alpha": [0.01, 0.001, 0.0001, 0.00001],   # how strongly the classifier is kept simple
    "clf__class_weight": [None, "balanced"],        # "balanced" counts each post of a small class for more
}

for score_name in ["accuracy", "f1_macro"]:
    search = GridSearchCV(pipeline, settings, scoring=score_name, cv=5)
    search.fit(train_posts, train_groups)
    print("Best settings when scoring by", score_name)
    print("  ", search.best_params_)
    print("   cross-validation", score_name, "=", round(search.best_score_, 3))
    print()
    tuned_groups = search.predict(test_posts)
    print(classification_report(test_groups, tuned_groups,
                                target_names=group_names))

Best settings when scoring by accuracy
   {'clf__alpha': 0.0001, 'clf__class_weight': 'balanced', 'vect__ngram_range': (1, 2)}
   cross-validation accuracy = 0.976

                    precision    recall  f1-score   support

     comp.graphics       0.91      0.97      0.94       389
  rec.sport.hockey       0.97      0.99      0.98       399
           sci.med       0.96      0.90      0.93       396
talk.religion.misc       0.82      0.72      0.77        50

          accuracy                           0.94      1234
         macro avg       0.91      0.89      0.90      1234
      weighted avg       0.94      0.94      0.94      1234



Best settings when scoring by f1_macro
   {'clf__alpha': 0.0001, 'clf__class_weight': 'balanced', 'vect__ngram_range': (1, 1)}
   cross-validation f1_macro = 0.966

                    precision    recall  f1-score   support

     comp.graphics       0.91      0.97      0.94       389
  rec.sport.hockey       0.98      0.99      0.98       399
           sci.med       0.96      0.89      0.93       396
talk.religion.misc       0.82      0.84      0.83        50

          accuracy                           0.94      1234
         macro avg       0.92      0.92      0.92      1234
      weighted avg       0.95      0.94      0.94      1234



Before tuning, in section 2, the classifier found 26 of the 50 religion posts (recall 0.52). Both tuned classifiers use a smaller `alpha` and `class_weight="balanced"`, and they find many more: 36 with the settings chosen by accuracy (recall 0.72) and 42 with the settings chosen by `f1_macro` (recall 0.84). The two scores disagree about word pairs. The F1 of the religion class went from 0.68 to 0.77 and 0.83, and the macro average F1 from 0.88 to 0.90 and 0.92. There is a cost: the precision of the religion class went down from 0.96 to 0.82, because the classifier now puts more posts in that class that don't belong there.

Now look at the accuracy on the test posts: it is 0.94 for both tuned classifiers, the same as before tuning. If accuracy were the only number you looked at, you would think tuning had done nothing. Keep in mind that there are only 50 religion posts in the test set, so the difference between the two tuned classifiers is 6 posts. It is a real difference but not a big one. The lesson is that the score you tune for decides which settings you get, so it should be the score that measures what you care about.

Notice that the test posts were not used to choose the settings. The grid search chose them with cross-validation on the training data, and the test data was used only at the end, to see how well the tuned classifiers work. (Both tuned classifiers are tested so that you can compare them.) If you chose the settings by looking at the test results, the test results would no longer tell you how well the classifier works on new posts.

## Try It Yourself

- Look at the confusion matrix. Which two newsgroups are confused most often? Calculate the precision and recall of `sci.med` from the matrix, and check your answers against the report.
- In the data code cell, change the 75 and the 50 to 377 and 251, so that all of the religion posts are used, and run the classifier cell again. How do the recall of `talk.religion.misc` and the macro average change? Run the grid search cell again: does tuning still make as much difference? Try 150 and 100 as well.

*This notebook was drafted by Claude Code and reviewed by Brian Bird. 10/21/2026*